In [1]:
import wave

In [2]:
import numpy as np
import pandas as pd
import wave
import math
import struct
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Activation

In [3]:
# Prepare a dummy data for simulating musical notes
notes_freqs = {
    'A': 440.0, 'B': 493.88, 'C': 261.63, 'D': 293.66, 'E': 393.63,
    'F': 349.23, 'G': 392.0
}

In [4]:
# View notes frequencies
notes_freqs

{'A': 440.0,
 'B': 493.88,
 'C': 261.63,
 'D': 293.66,
 'E': 393.63,
 'F': 349.23,
 'G': 392.0}

In [5]:
# Extract list of unique notes
notes = list(notes_freqs.keys())
notes

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [6]:
# Create mapping from note names to unique integers
note_to_int = {note: i for i, note in enumerate(notes)}
note_to_int

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}

In [7]:

# Create mapping from integers back to note names
int_to_note = {i: note for i, note in enumerate(notes)}
int_to_note

{0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

In [8]:
raw_music_data = [notes[np.random.randint(0,7)] for i in range(1000)]

In [9]:
print(raw_music_data)

['B', 'A', 'G', 'G', 'F', 'C', 'B', 'E', 'A', 'F', 'E', 'E', 'E', 'B', 'D', 'D', 'C', 'C', 'F', 'A', 'G', 'D', 'G', 'G', 'D', 'D', 'D', 'B', 'E', 'F', 'G', 'A', 'B', 'A', 'A', 'A', 'F', 'E', 'D', 'E', 'C', 'E', 'C', 'E', 'F', 'A', 'C', 'F', 'E', 'C', 'G', 'A', 'C', 'E', 'B', 'C', 'D', 'F', 'B', 'D', 'D', 'G', 'A', 'B', 'A', 'A', 'E', 'F', 'E', 'F', 'E', 'E', 'B', 'E', 'F', 'F', 'D', 'F', 'F', 'F', 'D', 'F', 'A', 'B', 'C', 'A', 'D', 'B', 'D', 'D', 'G', 'C', 'F', 'A', 'D', 'G', 'E', 'A', 'G', 'D', 'C', 'C', 'B', 'B', 'B', 'B', 'C', 'B', 'D', 'E', 'G', 'F', 'C', 'A', 'A', 'A', 'G', 'F', 'B', 'E', 'C', 'G', 'E', 'E', 'D', 'E', 'E', 'D', 'A', 'G', 'B', 'A', 'C', 'E', 'B', 'A', 'A', 'D', 'C', 'G', 'F', 'B', 'F', 'D', 'B', 'G', 'F', 'A', 'E', 'A', 'B', 'A', 'D', 'G', 'F', 'F', 'B', 'D', 'G', 'A', 'A', 'F', 'B', 'A', 'E', 'B', 'F', 'F', 'E', 'G', 'A', 'C', 'E', 'F', 'C', 'F', 'F', 'C', 'B', 'F', 'D', 'D', 'F', 'D', 'D', 'B', 'E', 'B', 'C', 'B', 'G', 'A', 'G', 'F', 'B', 'E', 'G', 'A', 'F', 'A',

#### PREPARE THE DATA

In [10]:
seq_length = 3
network_input = []
network_output = []

for i in range(len(raw_music_data) - seq_length):
    seq_in = raw_music_data[i: i+seq_length]
    seq_out = raw_music_data[i+seq_length]
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])
    print(seq_in, '-->', seq_out)

['B', 'A', 'G'] --> G
['A', 'G', 'G'] --> F
['G', 'G', 'F'] --> C
['G', 'F', 'C'] --> B
['F', 'C', 'B'] --> E
['C', 'B', 'E'] --> A
['B', 'E', 'A'] --> F
['E', 'A', 'F'] --> E
['A', 'F', 'E'] --> E
['F', 'E', 'E'] --> E
['E', 'E', 'E'] --> B
['E', 'E', 'B'] --> D
['E', 'B', 'D'] --> D
['B', 'D', 'D'] --> C
['D', 'D', 'C'] --> C
['D', 'C', 'C'] --> F
['C', 'C', 'F'] --> A
['C', 'F', 'A'] --> G
['F', 'A', 'G'] --> D
['A', 'G', 'D'] --> G
['G', 'D', 'G'] --> G
['D', 'G', 'G'] --> D
['G', 'G', 'D'] --> D
['G', 'D', 'D'] --> D
['D', 'D', 'D'] --> B
['D', 'D', 'B'] --> E
['D', 'B', 'E'] --> F
['B', 'E', 'F'] --> G
['E', 'F', 'G'] --> A
['F', 'G', 'A'] --> B
['G', 'A', 'B'] --> A
['A', 'B', 'A'] --> A
['B', 'A', 'A'] --> A
['A', 'A', 'A'] --> F
['A', 'A', 'F'] --> E
['A', 'F', 'E'] --> D
['F', 'E', 'D'] --> E
['E', 'D', 'E'] --> C
['D', 'E', 'C'] --> E
['E', 'C', 'E'] --> C
['C', 'E', 'C'] --> E
['E', 'C', 'E'] --> F
['C', 'E', 'F'] --> A
['E', 'F', 'A'] --> C
['F', 'A', 'C'] --> F
['A', 'C',

In [11]:
n_patterns = len(network_input)
n_patterns

997

In [12]:

X = np.reshape(network_input, (n_patterns, seq_length, 1))
X

array([[[1],
        [0],
        [6]],

       [[0],
        [6],
        [6]],

       [[6],
        [6],
        [5]],

       ...,

       [[6],
        [0],
        [0]],

       [[0],
        [0],
        [3]],

       [[0],
        [3],
        [5]]])

In [13]:
from keras.utils import to_categorical


In [14]:
y = to_categorical(network_output)

In [15]:
y

array([[0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], dtype=float32)

In [16]:
y.shape

(997, 7)

In [21]:
from tensorflow.keras.layers import Input

#### BUILD THE MODEL

In [31]:
model = Sequential()
model.add(Input((3,1)))
model.add(GRU(256))
model.add(Dense(512,activation='relu'))
model.add(Dense(7,activation='softmax'))

In [32]:
model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru_1 (GRU)                 (None, 256)               198912    
                                                                 
 dense_2 (Dense)             (None, 512)               131584    
                                                                 
 dense_3 (Dense)             (None, 7)                 3591      
                                                                 
Total params: 334087 (1.27 MB)
Trainable params: 334087 (1.27 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [33]:
model.compile(loss='categorical_crossentropy',optimizer='adam', metrics=['accuracy'])

In [34]:
model.fit(X,y,epochs=1000,batch_size=10)

Epoch 1/1000
100/100 [==============================] - 3s 7ms/step - loss: 1.9683 - accuracy: 0.1474
Epoch 2/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.9483 - accuracy: 0.1585
Epoch 3/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.9482 - accuracy: 0.1515
Epoch 4/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.9442 - accuracy: 0.1494
Epoch 5/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.9415 - accuracy: 0.1454
Epoch 6/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.9418 - accuracy: 0.1394
Epoch 7/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.9369 - accuracy: 0.1685
Epoch 8/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.9380 - accuracy: 0.1625
Epoch 9/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.9373 - accuracy: 0.1715
Epoch 10/1000
100/100 [==============================] - 1s 7ms/step - loss: 1.936

KeyboardInterrupt: 

#### GENERATE NEW MELODY SEQUENCE

In [29]:
start_index = np.random.randint(0, len(network_output))
pattern = network_input[start_index]
pattern

[6, 1, 6]

In [38]:
generated_melody = []

for i in range(8):
    x_input = np.reshape(pattern, (1, len(pattern), 1))
    pred = model.predict(x_input, verbose=False)
    index = np.argmax(pred)
    result = int_to_note[index]
    generated_melody.append(result)
    pattern.append(index)
    pattern = pattern[1:len(pattern)]

In [39]:
generated_melody

['B', 'G', 'B', 'G', 'B', 'G', 'B', 'G']

#### save this as audiofile

In [40]:
with wave.open('my_music.wav', 'w') as wav_file:
    wav_file.setparams((1, 2, 44100, 0, 'NONE', 'not compressed'))
    for note in generated_melody:
        freq = notes_freqs[note]
        num_samples = int(0.5 * 44100)
        for i in range(num_samples):
            t = float(i) / 44100
            value = int(32767 * 0.5 * math.sin(2 * math.pi * (freq * t)))
            data = struct.pack('<h', value)
            wav_file.writeframes(data)